In [16]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import sympy as sp
from scipy.integrate import odeint
sp.init_printing(use_latex=True)

In [17]:
def generar_base_24(limite):
    """
    Toma una recta numérica del 1 al 'limite' y la proyecta en el espacio Tetravigesimal.
    Retorna un diccionario con los números, su estado primo, y su posición en la Base 24.
    """
    numeros = np.arange(1, limite + 1)
    es_primo = np.array([sp.isprime(int(n)) for n in numeros])
    posicion_base_24 = numeros % 24
    return {
        "n": numeros,
        "es_primo": es_primo,
        "base_24": posicion_base_24
    }
datos = generar_base_24(100)
print(f"Número 97: ¿Es primo? {datos['es_primo'][96]} | Posición en Base 24: {datos['base_24'][96]}")

Número 97: ¿Es primo? True | Posición en Base 24: 1


In [18]:
t = np.linspace(0, 20, 1000)
x = t * np.cos(t)
y = t * np.sin(t)
z = t
fig = go.Figure(data=[go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color='orange', width=4))])
fig.update_layout(
    title='Atractor de Prueba (Topología Base 24)',
    scene=dict(xaxis_title='Eje X', yaxis_title='Eje Y', zaxis_title='Eje Z'),
    template='plotly_dark'
)
fig.show()

In [19]:
limite = 500
paso = 5
n_vals = np.arange(1, limite + 1)
primos = np.array([sp.isprime(int(i)) for i in n_vals])
angulos = (n_vals % 24) * 15
radios = (n_vals / 24) + 5
fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=[], theta=[], mode='markers',
    marker=dict(color='rgba(255, 255, 255, 0.15)', size=4),
    name='No Primo', hoverinfo='text'
))
fig.add_trace(go.Scatterpolar(
    r=[], theta=[], mode='markers',
    marker=dict(color='#FF8C00', size=8, symbol='circle', line=dict(color='white', width=0.5)),
    name='Primo (Atractor)', hoverinfo='text'
))
frames = []
for k in range(paso, limite + paso, paso):
    k = min(k, limite)
    idx = np.arange(0, k)
    idx_no_primos = idx[~primos[idx]]
    idx_primos = idx[primos[idx]]
    frame = go.Frame(
        data=[
            go.Scatterpolar(
                r=radios[idx_no_primos], theta=angulos[idx_no_primos],
                text=[f"N: {n_vals[i]}" for i in idx_no_primos]
            ),
            go.Scatterpolar(
                r=radios[idx_primos], theta=angulos[idx_primos],
                text=[f"N: {n_vals[i]} [PRIMO]" for i in idx_primos]
            )
        ],
        name=str(k)
    )
    frames.append(frame)
fig.frames = frames
sliders = [{
    "pad": {"b": 10, "t": 50},
    "len": 0.9, "x": 0.1, "y": 0,
    "currentvalue": {"font": {"size": 18, "color": "#FF8C00"}, "prefix": "Límite actual N = ", "visible": True, "xanchor": "right"},
    "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}], "label": f.name, "method": "animate"} for f in frames]
}]
fig.update_layout(
    title='Formación de los 8 Atractores (Base Guzmánica)',
    template='plotly_dark',
    width=800,
    height=800,
    polar=dict(
        angularaxis=dict(tickmode='array', tickvals=np.arange(0, 360, 15), ticktext=np.arange(0, 24)),
        radialaxis=dict(visible=False, range=[0, (limite/24) + 6])
    ),
    updatemenus=[{
        "buttons": [
            {"args": [None, {"frame": {"duration": 150, "redraw": True}, "fromcurrent": True}], "label": "▶ Reproducir", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}], "label": "⏸ Pausa", "method": "animate"}
        ],
        "direction": "left", "pad": {"r": 10, "t": 87}, "showactive": False, "type": "buttons", "x": 0.1, "xanchor": "right", "y": 0, "yanchor": "top"
    }],
    sliders=sliders
)
fig.show()